# 07 · Six-modality fusion built on the per-modality notebooks

Layer 1 is **not re-trained here**. It is read from the six modality notebooks (`01_proteomics … 06_cnv`), which
each re-ran their own feature selection and circuit on the **same site-held-out split and the same 5 site-grouped
folds**. This notebook checks that all six used the same split (split hash), then runs the unchanged Layer 2:
mean · reliability-weighted · learned modality weights (global / per class) · stacking LR · meta-VQC + matched LR ·
hierarchical, scored once on the 7 held-out hospitals with bootstrap CIs.

| Base family | What it is |
|---|---|
| `LR` | tuned logistic regression per modality (identical to the earlier fusion notebook) |
| `VQC` | **each modality notebook's own VQC** (its feature selection, qubit count and circuit) |
| `LRm` | logistic regression on exactly the VQC's features (matched classical baseline; set `INCLUDE_LR_MATCHED = True` to fuse it too) |

**Inputs:** add the output of each of the six modality notebooks as an input of this notebook
(*Add input → Notebook output*). The files are found automatically under `/kaggle/input`.
No raw data are read, so this notebook runs in ≈ 10–20 min (most of it the meta-VQC; `RUN_META_VQC = False` → ≈ 3 min).

**Primary model:** learned modality weights (global) on the `LR` base, as before (`PRIMARY_MODEL` in the config).

In [1]:
# ============================================================
# 0. IMPORTS
# ============================================================
import os
os.environ["PYTHONWARNINGS"] = "ignore"          # also silences warnings in parallel workers
import re, csv, time, json, warnings, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score, log_loss,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)
csv.field_size_limit(2**31 - 1)
try:
    display
except NameError:
    display = print

In [2]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
# FILES = {
#     "proteomics":      "/kaggle/input/datasets/devipriyabala/unbalanced/proteomics_matrix.csv",
#     "methylation":     "/kaggle/input/datasets/devipriyabala/unbalanced/methylation_matrix.csv",
#     "gene_expression": "/kaggle/input/datasets/devipriyabala/unbalanced/gene_expr_matrix.csv",
#     "mirna":           "/kaggle/input/datasets/devipriyabala/unbalanced/mirna_matrix.csv",
#     "snv":             "/kaggle/input/datasets/nirupamas1746/snv-dataset/snv1.csv",
#     # alternative SNV file (no caller column -> TMB left out):
#     # "snv":           "/kaggle/input/datasets/devipriyabala/unbalanced/snv_matrix.csv",
#     "cnv":             "/kaggle/input/datasets/nirupamas1746/cnv-fixed/cnv_fixed.csv",
# }
FILES = {
    "proteomics":      "/kaggle/input/datasets/devipriyabala/unbalanced/proteomics_matrix.csv",
    "methylation":     "/kaggle/input/datasets/devipriyabala/unbalanced/methylation_matrix.csv",
    "gene_expression": "/kaggle/input/datasets/devipriyabala/unbalanced/gene_expr_matrix.csv",
    "mirna":           "/kaggle/input/datasets/devipriyabala/unbalanced/mirna_matrix.csv",
    "snv":             "/kaggle/input/datasets/nirupamas1746/snv-dataset/snv1.csv",
    "cnv":             "/kaggle/input/datasets/nirupamas1746/cnv-fixed/cnv_fixed.csv",
}

for _m in FILES:                                               # optional overrides, e.g. CNV_FILE=...
    FILES[_m] = os.environ.get(f"{_m.upper()}_FILE", FILES[_m])
missing_files = [p for p in FILES.values() if not os.path.exists(p)]
# raw files are not needed here: Layer 1 comes from the modality notebooks
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODALITIES = list(FILES)
SHORT_M = {"proteomics": "PROT", "methylation": "METH", "gene_expression": "GE", "mirna": "miRNA",
           "snv": "SNV", "cnv": "CNV"}
LABEL_COL = "subtype"
CLASSES = ["astrocytoma", "glioblastoma", "mixed_glioma", "oligodendroglioma"]
K = len(CLASSES)
GBM, MIXED = "glioblastoma", "mixed_glioma"
LGG = [c for c in CLASSES if c != GBM]
SHORT = ["Astro", "GBM", "Mixed", "Oligo"]

RANDOM_STATE = 42
TEST_SITES = ["12", "14", "27", "CS", "DU", "QH", "RY"]
OUTER_FOLDS, INNER_FOLDS = 5, 4
N_BOOT = 2000
NA_STR = {"", "nan", "NaN", "NA", "na", "NULL", "null", "None"}

# --- modality preprocessing (all label-free, fitted on training patients only) ---
PROT_SAMPLE_MISSING = 0.20      # drop old-panel proteomics samples (~52% missing, all GBM)
PROT_MAX_MISSING = 0.05         # drop a protein if >5% missing among training patients
USE_PROBE_BLACKLIST = True
METH_PRESENT_PER_PLATFORM = 0.90  # probe kept only if present in >=90% of training samples of EACH array platform
METH_TOP_VAR = 10000
M_EPSILON = 1e-3
GE_EXPR_MIN, GE_EXPR_FRAC, GE_TOP_VAR = 1.0, 0.20, 5000
MIRNA_DETECT_FRAC = 0.20
SNV_MIN_FREQ = 0.02
SNV_DRIVERS = ["EGFR", "PTEN", "NF1", "IDH1", "IDH2", "TP53", "ATRX", "CIC", "FUBP1", "NOTCH1"]
SNV_META = ["patient_folder", "Tumor_Sample_Barcode", "n_caller_files"]
TMB_BINS = 4
CNV_REGION_R = 0.90

# --- base learners: grids tuned by inner site-grouped CV ---
BASE_GRIDS = {
    "proteomics":      {"sel__spec": [("anova", 15), ("anova", 30), ("anova", 60), ("hier", 5, 25)],
                        "clf__C": [0.03, 0.3, 1.0]},
    "methylation":     {"sel__spec": [("anova", 10), ("anova", 50), ("anova", 200), ("hier", 2, 10), ("hier", 10, 90)],
                        "clf__C": [0.01, 0.1, 1.0]},
    "gene_expression": {"sel__spec": [("anova", 10), ("anova", 50), ("anova", 200), ("hier", 2, 10), ("hier", 10, 90)],
                        "clf__C": [0.01, 0.1, 1.0]},
    "mirna":           {"sel__spec": [("anova", 10), ("anova", 30), ("anova", 100), ("hier", 5, 25)],
                        "clf__C": [0.03, 0.3]},
    "snv":             {"sel__spec": [("drivers",), ("drivers+anova", 20), ("anova", 50)],
                        "clf__C": [0.1, 1.0]},
    "cnv":             {"sel__spec": [("anova", 6), ("anova", 12), ("anova", 24), ("all",)],
                        "clf__C": [0.03, 0.3, 1.0]},
}

# --- quantum models (exact statevector, adjoint gradients; engine from the SNV+CNV fusion notebook) ---
RUN_BASE_VQC = True
BASE_VQC_LAYERS, BASE_VQC_EPOCHS, BASE_VQC_LR, BASE_VQC_SEEDS = 2, 60, 0.05, (1, 2, 3)
BASE_VQC_OVERRIDE = {"snv": dict(epochs=45, lr=0.08)}   # SNV notebook's CV-chosen settings (11 qubits)
RUN_META_VQC = True
# Primary (reported) fusion model. Named on design grounds: interpretable per-modality weights, the method the
# study was built around. Set to None to use the highest-CV fusion method instead. The CV-argmax method is
# always reported next to it.
# PRIMARY_MODEL = ("LR", "Learned weights (global)")
PRIMARY_MODEL = ("VQC", "Learned weights (global)")
RUN_HIERARCHICAL = True         # LGG-specific base models + hierarchical fusion (adds ~50% to the classical base time)
META_VQC_K = 8                  # meta-features -> 8 qubits (24 meta-features = 6 modalities x 4 classes)
META_LAYERS, META_EPOCHS, META_LR, META_SEEDS = 2, 60, 0.05, (1, 2, 3)

# --- inputs from the six modality notebooks ---
PROB_ROOT = os.environ.get("PROB_ROOT", "/kaggle/input" if os.path.isdir("/kaggle/input") else ".")
INCLUDE_LR_MATCHED = False      # True: also fuse the matched-feature logistic regression (LRm) base

## 2. Layer 1 from the modality notebooks

In [3]:
# ============================================================
# 2. LOAD LAYER 1 FROM THE SIX MODALITY NOTEBOOKS (and check they share one split)
# ============================================================
import glob


def _find(name):
    return sorted(glob.glob(os.path.join(PROB_ROOT, "**", name), recursive=True))


paths, INFO = {}, {}
for m in MODALITIES:
    hits = _find(f"probs_{m}.csv")
    assert hits, f"probs_{m}.csv not found under {PROB_ROOT}: add the {m} notebook's output as an input"
    if len(hits) > 1:
        print(f"several probs_{m}.csv found, using {hits[0]}")
    paths[m] = hits[0]
    INFO[m] = json.load(open(os.path.join(os.path.dirname(hits[0]), f"info_{m}.json")))
hashes = {m: INFO[m]["split_hash"] for m in MODALITIES}
assert len(set(hashes.values())) == 1, f"The modality notebooks used different splits: {hashes}"
print("All six modality notebooks used split", next(iter(hashes.values())))

cohort = pd.read_csv(os.path.join(os.path.dirname(paths[MODALITIES[0]]), "split_shared.csv"), index_col=0)
cohort.index = cohort.index.astype(str)
cohort = cohort.rename(columns={f"has_{m}": m for m in MODALITIES})
for m in MODALITIES:
    cohort[m] = cohort[m].astype(bool)
mi = _find("meth_info.csv")
METH_INFO = (pd.read_csv(mi[0], index_col=0) if mi else pd.DataFrame(columns=["subtype", "platform"]))
METH_INFO.index = METH_INFO.index.astype(str)
cohort["meth_platform"] = METH_INFO["platform"].reindex(cohort.index).fillna("none")
cohort["n_modalities"] = cohort[MODALITIES].sum(axis=1)
cohort["complete"] = cohort["n_modalities"] == len(MODALITIES)
TRAIN = cohort.index[cohort["split"] == "train"]
TEST = cohort.index[cohort["split"] == "test"]
FOLD = cohort.loc[TRAIN, "fold"].astype(int)
assert set(cohort.loc[TRAIN, "site"]).isdisjoint(TEST_SITES)

BASE_FAMILIES = ["LR", "VQC"] + (["LRm"] if INCLUDE_LR_MATCHED else [])
OOF = {f: {} for f in ["LR", "VQC", "LRm"]}
TESTP = {f: {} for f in ["LR", "VQC", "LRm"]}
OOF_LGG, TESTP_LGG = {}, {}
for m in MODALITIES:
    P = pd.read_csv(paths[m]); P["patient"] = P["patient"].astype(str)
    get = lambda model, st, cols: P[(P.model == model) & (P.set == st)].set_index("patient")[cols].astype(float)
    for f in ["LR", "VQC", "LRm"]:
        OOF[f][m], TESTP[f][m] = get(f, "oof", CLASSES), get(f, "test", CLASSES)
    OOF_LGG[m], TESTP_LGG[m] = get("LR_LGG", "oof", LGG), get("LR_LGG", "test", LGG)
    assert set(OOF["LR"][m].index) == set(TRAIN[cohort.loc[TRAIN, m].values]), f"{m}: training patients differ"
BASE_INFO = [{k: INFO[m][k] for k in ["modality", "n_train", "n_test", "final_params", "vqc_qubits", "vqc_features",
                                      "vqc_n_params"]} for m in MODALITIES]


def macro(y, pred, labels=CLASSES):
    return f1_score(y, pred, labels=labels, average="macro", zero_division=0)


rows = []
for m in MODALITIES:
    for f in ["VQC", "LRm", "LR"]:
        r = {"modality": SHORT_M[m], "base": f, "qubits": INFO[m]["vqc_qubits"] if f != "LR" else "-"}
        for ev, D in [("CV", OOF), ("test", TESTP)]:
            d = D[f][m]; r[f"{ev}_macro_f1"] = macro(cohort.loc[d.index, "subtype"], d.idxmax(axis=1))
        rows.append(r)
BASE_TABLE = pd.DataFrame(rows)
print("Per-modality base models on the shared split (VQC = each modality notebook's own circuit):")
display(BASE_TABLE.round(3))
print(f"Patients: train {len(TRAIN)} | test {len(TEST)} | complete-case test {int(cohort.loc[TEST, 'complete'].sum())}")

All six modality notebooks used split f3ea2f6bb5ee
Per-modality base models on the shared split (VQC = each modality notebook's own circuit):


,modality,base,qubits,CV_macro_f1,test_macro_f1
0,PROT,VQC,6,0.528,0.521
1,PROT,LRm,6,0.580,0.530
2,PROT,LR,-,0.550,0.584
3,METH,VQC,4,0.540,0.561
4,METH,LRm,4,0.518,0.575
5,METH,LR,-,0.555,0.705
6,GE,VQC,9,0.547,0.670
7,GE,LRm,9,0.631,0.668
8,GE,LR,-,0.615,0.588
9,miRNA,VQC,8,0.594,0.614


Patients: train 315 | test 124 | complete-case test 81


Statevector engine for the meta-VQC (Layer 2).

In [4]:
# ------------------------------------------------------------
# Exact statevector engine (verbatim from 03_Fusion_SNV_CNV.ipynb) + a gradient self-check
# ------------------------------------------------------------
class Engine:
    def __init__(self, n, layers, reupload=True):
        self.n, self.dim = n, 2 ** n
        bits = (np.arange(self.dim)[:, None] >> np.arange(n)) & 1
        self.zsign = (1 - 2 * bits).astype(float)
        pairs = [(q, q + 1) for q in range(0, n - 1, 2)] + [(q, q + 1) for q in range(1, n - 1, 2)]
        cz = np.ones(self.dim)
        for c, t in pairs:
            cz *= np.where(bits[:, c] & bits[:, t], -1.0, 1.0)
        self.cz = cz
        ops, k = [], 0
        for layer in range(layers):
            if layer == 0 or reupload:
                ops += [("enc", q) for q in range(n)]
            for q in range(n):
                ops.append(("ry", k, q)); k += 1
                ops.append(("rz", k, q)); k += 1
            ops.append(("cz",))
        self.ops, self.P = ops, k

    def _ap(self, psi, a, b, c, d, q):
        B = psi.shape[0]; v = psi.reshape(B, -1, 2, 1 << q)
        a0, a1 = v[:, :, 0, :], v[:, :, 1, :]
        o = np.empty_like(v); o[:, :, 0, :] = a * a0 + b * a1; o[:, :, 1, :] = c * a0 + d * a1
        return o.reshape(B, -1)
    def ry(self, psi, t, q):
        c, s = np.cos(t / 2), np.sin(t / 2)
        if np.ndim(t): c, s = c[:, None, None], s[:, None, None]
        return self._ap(psi, c, -s, s, c, q)
    def rz(self, psi, t, q):
        e = np.exp(-0.5j * t); return self._ap(psi, e, 0, 0, np.conj(e), q)
    def dry(self, psi, t, q):
        c, s = np.cos(t / 2) / 2, np.sin(t / 2) / 2; return self._ap(psi, -s, -c, c, -s, q)
    def drz(self, psi, t, q):
        e = np.exp(-0.5j * t); return self._ap(psi, -0.5j * e, 0, 0, 0.5j * np.conj(e), q)

    def state(self, Xa, th, shift_occ=-1, delta=0.0):
        # forward pass; optionally shift ONE input-encoding occurrence (used by Q-SHAP)
        psi = np.zeros((len(Xa), self.dim), complex); psi[:, 0] = 1; n_enc = 0
        for op in self.ops:
            if op[0] == "enc":
                psi = self.ry(psi, Xa[:, op[1]] + (delta if n_enc == shift_occ else 0.0), op[1]); n_enc += 1
            elif op[0] == "ry": psi = self.ry(psi, th[op[1]], op[2])
            elif op[0] == "rz": psi = self.rz(psi, th[op[1]], op[2])
            else: psi = psi * self.cz
        return psi
    def expz(self, psi):
        return (np.abs(psi) ** 2) @ self.zsign
    def grad(self, Xa, th, psi, G):
        phi, lam = psi.copy(), psi * (G @ self.zsign.T); g = np.zeros(self.P)
        for op in reversed(self.ops):
            if op[0] == "cz": phi, lam = phi * self.cz, lam * self.cz
            elif op[0] == "enc":
                a = Xa[:, op[1]]; phi, lam = self.ry(phi, -a, op[1]), self.ry(lam, -a, op[1])
            else:
                kind, k, q = op; t = th[k]
                if kind == "ry": phi = self.ry(phi, -t, q); mu, lam2 = self.dry(phi, t, q), self.ry(lam, -t, q)
                else:            phi = self.rz(phi, -t, q); mu, lam2 = self.drz(phi, t, q), self.rz(lam, -t, q)
                g[k] = 2 * np.real(np.vdot(lam, mu)); lam = lam2
        return g

def unique_patterns(A):
    u, inv = np.unique(A, axis=0, return_inverse=True); return u.astype(float), np.asarray(inv).ravel()

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def train_vqc(F, y, layers, epochs, lr, seed, init_std=0.1):
    eng = Engine(F.shape[1], layers); rng = np.random.default_rng(seed)
    Xu, inv = unique_patterns(F); Xa = Xu * np.pi; N = len(y)
    sw = (N / (K * np.bincount(y, minlength=K)))[y][:, None]; Yo = np.eye(K)[y]
    q = rng.normal(0, init_std, eng.P); W = rng.normal(0, init_std, (K, eng.n)); b = np.zeros(K)
    P = [q, W, b]; m = [np.zeros_like(p) for p in P]; v = [np.zeros_like(p) for p in P]
    for ep in range(1, epochs + 1):
        psi = eng.state(Xa, q); Zu = eng.expz(psi); Z = Zu[inv]
        pr = softmax(Z @ W.T + b); dl = sw * (pr - Yo) / N
        dZu = np.zeros_like(Zu); np.add.at(dZu, inv, dl @ W)
        for i, g in enumerate([eng.grad(Xa, q, psi, dZu), dl.T @ Z, dl.sum(0)]):
            m[i] = 0.9 * m[i] + 0.1 * g; v[i] = 0.999 * v[i] + 0.001 * g * g
            P[i] -= lr * (m[i] / (1 - 0.9 ** ep)) / (np.sqrt(v[i] / (1 - 0.999 ** ep)) + 1e-8)
    return eng, (q.copy(), W.copy(), b.copy())

def vqc_proba(eng, params, F):
    q, W, b = params; Xu, inv = unique_patterns(F)
    return softmax(eng.expz(eng.state(Xu * np.pi, q))[inv] @ W.T + b)


_rng = np.random.default_rng(0)
_e = Engine(5, 2)
_th = _rng.uniform(-np.pi, np.pi, _e.P)
_Xa = _rng.uniform(0, np.pi, (4, 5))
_psi = _e.state(_Xa, _th)
_G = _rng.normal(size=(4, 5))
_ga = _e.grad(_Xa, _th, _psi, _G)
_gp = np.array([np.sum(_G * (_e.expz(_e.state(_Xa, _th + d)) - _e.expz(_e.state(_Xa, _th - d))) / 2)
                for d in np.eye(_e.P) * (np.pi / 2)])
print(f"adjoint vs parameter-shift gradient, max diff: {np.abs(_ga - _gp).max():.1e}")
assert np.abs(_ga - _gp).max() < 1e-8

adjoint vs parameter-shift gradient, max diff: 1.6e-15


In [5]:
# ------------------------------------------------------------
# How good is each modality on its own, and how complementary are they?
# ------------------------------------------------------------
def gbm_recall(y, pred, plat, p):
    msk = (y == GBM) & (plat == p)
    return float((pred[msk] == GBM).mean()) if msk.sum() else np.nan


rows = []
for fam in BASE_FAMILIES:
    for m in MODALITIES:
        for ev, P_ in [("CV", OOF[fam][m]), ("TEST", TESTP[fam][m])]:
            y_ = cohort.loc[P_.index, "subtype"]; pr = P_.idxmax(1); pl_ = cohort.loc[P_.index, "meth_platform"]
            f = f1_score(y_, pr, labels=CLASSES, average=None, zero_division=0)
            lg = y_ != GBM
            rows.append({"base": fam, "modality": SHORT_M[m], "eval": ev, "n": len(y_), "macro_f1": macro(y_, pr),
                         "lgg_only_macro_f1": macro(y_[lg], pr[lg], LGG),
                         **{f"f1_{s}": v for s, v in zip(SHORT, f)},
                         "gbm_recall_27K": gbm_recall(y_, pr, pl_, "27K"), "gbm_recall_450K": gbm_recall(y_, pr, pl_, "450K")})
MOD_TABLE = pd.DataFrame(rows)
display(MOD_TABLE.round(3))
MOD_TABLE.to_csv(os.path.join(OUTPUT_DIR, "per_modality_results.csv"), index=False)

# complementarity on training patients with all modalities (LR base)
cc = cohort.loc[TRAIN][cohort.loc[TRAIN, "complete"]].index
correct = pd.DataFrame({SHORT_M[m]: OOF["LR"][m].loc[cc].idxmax(1) == cohort.loc[cc, "subtype"] for m in MODALITIES})
print(f"\nComplete-case training patients: {len(cc)}")
print("Accuracy per modality:", correct.mean().round(3).to_dict())
print(f"At least one modality correct (oracle ceiling): {correct.any(axis=1).mean():.3f} | "
      f"majority of modalities correct: {(correct.sum(axis=1) > len(MODALITIES) / 2).mean():.3f}")
print("Pairwise agreement of errors (phi correlation of 'wrong' indicators):")
display((~correct).astype(float).corr().round(2))

,base,modality,eval,n,macro_f1,lgg_only_macro_f1,f1_Astro,f1_GBM,f1_Mixed,f1_Oligo,gbm_recall_27K,gbm_recall_450K
0,LR,PROT,CV,223,0.550,0.490,0.500,0.754,0.435,0.509,0.895,0.600
1,LR,PROT,TEST,95,0.584,0.535,0.615,0.733,0.385,0.605,1.000,1.000
2,LR,METH,CV,303,0.555,0.497,0.550,0.778,0.457,0.436,1.000,0.000
3,LR,METH,TEST,123,0.705,0.651,0.658,0.885,0.562,0.714,1.000,0.000
4,LR,GE,CV,313,0.615,0.501,0.639,0.970,0.432,0.419,0.976,0.875
5,LR,GE,TEST,123,0.588,0.451,0.444,1.000,0.343,0.565,1.000,1.000
6,LR,miRNA,CV,302,0.669,0.569,0.656,0.974,0.533,0.512,1.000,0.875
7,LR,miRNA,TEST,111,0.694,0.592,0.676,1.000,0.500,0.600,1.000,1.000
8,LR,SNV,CV,313,0.520,0.491,0.407,0.658,0.458,0.557,0.786,0.500
9,LR,SNV,TEST,123,0.510,0.474,0.275,0.635,0.478,0.651,0.889,1.000



Complete-case training patients: 208
Accuracy per modality: {'PROT': 0.51, 'METH': 0.471, 'GE': 0.538, 'miRNA': 0.601, 'SNV': 0.462, 'CNV': 0.486}
At least one modality correct (oracle ceiling): 0.899 | majority of modalities correct: 0.418
Pairwise agreement of errors (phi correlation of 'wrong' indicators):


,PROT,METH,GE,miRNA,SNV,CNV
PROT,1.00,0.23,0.27,0.22,0.23,0.20
METH,0.23,1.00,0.35,0.34,0.17,0.20
GE,0.27,0.35,1.00,0.45,0.06,0.13
miRNA,0.22,0.34,0.45,1.00,0.22,0.20
SNV,0.23,0.17,0.06,0.22,1.00,0.18
CNV,0.20,0.20,0.13,0.20,0.18,1.00


## 7. Layer 2 — fusion methods (all handle missing modalities)
| Method | Weights / parameters learned from training-site OOF probabilities |
|---|---|
| Best single modality | none |
| Mean | none (average of available modalities) |
| Reliability-weighted | per-modality, per-class weights = OOF per-class F1 |
| **Learned weights (global)** | one non-negative weight per modality in a log-linear pool, p(c) ∝ Π_m p_m(c)^{w_m}·e^{b_c}; fitted by minimising class-weighted log-loss with an L2 pull toward equal weights; missing modalities simply drop out of the product |
| **Learned weights (per class)** | as above with one weight per modality and class (24 weights, stronger L2) |
| Stacking LR | logistic regression on log-probabilities + missing-modality indicators, trained with modality dropout |
| **Meta-VQC** | 8 qubits; meta-features = the 8 most informative of the 24 probability inputs (mRMR inside each fold); missing modality = all-zero block; modality dropout (one random present modality masked per extra copy); RY(π·p) re-uploading, 2 layers, 60 epochs, 3 seeds |
| LR on the meta-VQC's 8 inputs | matched classical baseline for the meta-VQC |

In [8]:
def meta_arrays(Pdict, ids, lgg=None):
    """Probabilities (n x 4 per modality, 0.25 where missing) and presence masks.
    If LGG-specific probabilities are given they are added under keys 'lgg:<modality>' (n x 3)."""
    P, M = {}, {}
    for m in MODALITIES:
        d = Pdict[m].reindex(ids)
        M[m] = d.notna().all(axis=1).values
        P[m] = np.where(M[m][:, None], d.fillna(1.0 / K).values, 1.0 / K)
        if lgg is not None:
            d = lgg[m].reindex(ids)
            M["lgg:" + m] = d.notna().all(axis=1).values
            P["lgg:" + m] = np.where(M["lgg:" + m][:, None], d.fillna(1.0 / 3).values, 1.0 / 3)
    return P, M


def lgg_for(fam):
    return OOF_LGG if (fam == "LR" and RUN_HIERARCHICAL) else None


def lgg_test_for(fam):
    return TESTP_LGG if (fam == "LR" and RUN_HIERARCHICAL) else None


sub = lambda D, i: {k: v[i] for k, v in D.items()}
EPS = 1e-6


def class_weights(y, k=K):
    cnt = np.bincount(y, minlength=k)
    return (len(y) / (k * np.maximum(cnt, 1)))[y]


class Single:
    def __init__(self, m): self.m = m
    def fit(self, y, P, M): return self
    def proba(self, P, M):
        q = np.where(M[self.m][:, None], P[self.m], np.nan)
        # patients without this modality: fall back to the mean of the others (reported separately below)
        others = Mean().proba(P, M)
        return np.where(np.isnan(q), others, q)


class Mean:
    def fit(self, y, P, M): return self
    def proba(self, P, M):
        num = sum(P[m] * M[m][:, None] for m in MODALITIES)
        den = sum(M[m][:, None].astype(float) for m in MODALITIES)
        return num / np.maximum(den, 1)


class Reliability:
    def fit(self, y, P, M):
        self.w = {m: np.maximum(f1_score(y[M[m]], P[m][M[m]].argmax(1), labels=range(K), average=None,
                                         zero_division=0), 1e-3) for m in MODALITIES}
        return self
    def proba(self, P, M):
        num = sum(P[m] * self.w[m] * M[m][:, None] for m in MODALITIES)
        den = sum(self.w[m] * M[m][:, None] for m in MODALITIES)
        q = num / np.maximum(den, 1e-9)
        return q / q.sum(1, keepdims=True)


class LearnedWeights:
    """Log-linear pool: logit_c = sum_m mask_m * w_{m(,c)} * log p_m(c) + b_c."""
    def __init__(self, per_class=False, l2=None, k=K):
        self.per_class, self.k = per_class, k
        self.l2 = (0.1 if per_class else 0.05) if l2 is None else l2   # L2 on log-weights (pull toward 1)

    def _logits(self, theta, L, Mk):
        nm, k = len(MODALITIES), self.k
        W = np.exp(theta[:nm * k].reshape(nm, k)) if self.per_class else np.repeat(np.exp(theta[:nm])[:, None], k, 1)
        b = theta[-k:]
        return np.einsum("imk,mk->ik", L * Mk[:, :, None], W) + b

    def fit(self, y, P, M):
        k = self.k
        L = np.stack([np.log(P[m] + EPS) for m in MODALITIES], 1)
        Mk = np.stack([M[m] for m in MODALITIES], 1).astype(float)
        sw = class_weights(y, k); Y = np.eye(k)[y]
        nw = len(MODALITIES) * (k if self.per_class else 1)

        def obj(theta):
            z = self._logits(theta, L, Mk); z = z - z.max(1, keepdims=True)
            lp = z - np.log(np.exp(z).sum(1, keepdims=True))
            return -(sw * (Y * lp).sum(1)).sum() / sw.sum() + self.l2 * np.sum(theta[:nw] ** 2)

        th0 = np.zeros(nw + k)
        self.theta_ = minimize(obj, th0, method="L-BFGS-B").x
        return self

    def weights(self):
        nm = len(MODALITIES)
        if self.per_class:
            return pd.DataFrame(np.exp(self.theta_[:nm * self.k].reshape(nm, self.k)),
                                index=[SHORT_M[m] for m in MODALITIES], columns=SHORT if self.k == K else SHORT[:self.k])
        return pd.Series(np.exp(self.theta_[:nm]), index=[SHORT_M[m] for m in MODALITIES])

    def proba(self, P, M):
        L = np.stack([np.log(P[m] + EPS) for m in MODALITIES], 1)
        Mk = np.stack([M[m] for m in MODALITIES], 1).astype(float)
        z = self._logits(self.theta_, L, Mk); z = z - z.max(1, keepdims=True)
        e = np.exp(z); return e / e.sum(1, keepdims=True)


def dropout_augment(y, P, M, one_random=False, seed=RANDOM_STATE):
    """Original rows + copies with one present modality masked, so the meta-learner learns to cope with missing data.
    one_random=False: one copy per present modality (up to 6 copies); True: one copy, random modality (faster)."""
    n_present = sum(M[o].astype(int) for o in MODALITIES)
    ys, Ps, Ms = [y], [P], [M]
    if one_random:
        rng = np.random.default_rng(seed)
        Mk = np.stack([M[o] for o in MODALITIES], 1)
        keep = np.where(n_present > 1)[0]
        drop = np.array([rng.choice(np.where(Mk[i])[0]) for i in keep], dtype=int)
        Md = {o: M[o][keep].copy() for o in MODALITIES}
        for j, o in enumerate(MODALITIES):
            Md[o][drop == j] = False
        ys.append(y[keep]); Ps.append(sub(P, keep)); Ms.append(Md)
    else:
        for m in MODALITIES:
            keep = M[m] & (n_present > 1)
            if keep.sum() == 0:
                continue
            Md = {o: (M[o][keep] if o != m else np.zeros(keep.sum(), bool)) for o in MODALITIES}
            ys.append(y[keep]); Ps.append(sub(P, keep)); Ms.append(Md)
    cat = lambda Ds: {o: np.concatenate([d[o] for d in Ds]) for o in MODALITIES}
    return np.concatenate(ys), cat(Ps), cat(Ms)


def stack_features(P, M):
    blocks = []
    for m in MODALITIES:
        blocks.append(np.where(M[m][:, None], np.log(P[m] + EPS) - np.log(1.0 / K), 0.0))
        blocks.append((~M[m]).astype(float)[:, None])
    return np.hstack(blocks)


class StackLR:
    def __init__(self, C=0.3): self.C = C
    def fit(self, y, P, M):
        ya, Pa, Ma = dropout_augment(y, P, M)
        self.lr = LogisticRegression(max_iter=5000, class_weight="balanced", C=self.C).fit(stack_features(Pa, Ma), ya)
        return self
    def proba(self, P, M):
        p = self.lr.predict_proba(stack_features(P, M))
        out = np.zeros((len(p), K)); out[:, self.lr.classes_] = p
        return out


def meta_blocks(P, M):
    return np.column_stack([np.where(M[m][:, None], P[m], 0.0) for m in MODALITIES])


META_NAMES = [f"{SHORT_M[m]}:{s}" for m in MODALITIES for s in SHORT]


def mrmr_select(F, y, k):
    rel = np.nan_to_num(f_classif(F, y)[0], nan=0.0)
    C = np.abs(np.nan_to_num(np.corrcoef(F.T)))
    sel = [int(np.argmax(rel))]
    while len(sel) < min(k, F.shape[1]):
        score = rel / (C[:, sel].mean(1) + 1e-3)
        score[sel] = -np.inf
        sel.append(int(np.argmax(score)))
    return sel


class MetaVQC:
    def __init__(self, k=META_VQC_K, classical=False):
        self.k, self.classical = k, classical
    def fit(self, y, P, M):
        ya, Pa, Ma = dropout_augment(y, P, M, one_random=True)
        F = meta_blocks(Pa, Ma)
        self.sel_ = mrmr_select(F, ya, self.k)
        if self.classical:
            self.lr_ = LogisticRegression(max_iter=5000, class_weight="balanced").fit(F[:, self.sel_], ya)
        else:
            self.models_ = [train_vqc(F[:, self.sel_], ya, META_LAYERS, META_EPOCHS, META_LR, s) for s in META_SEEDS]
        return self
    def features(self, P, M):
        return meta_blocks(P, M)[:, self.sel_]
    def proba(self, P, M):
        F = self.features(P, M)
        if self.classical:
            p = self.lr_.predict_proba(F); out = np.zeros((len(p), K)); out[:, self.lr_.classes_] = p
            return out
        return np.mean([vqc_proba(e, prm, F) for e, prm in self.models_], axis=0)
    def feature_names(self):
        return [META_NAMES[j] for j in self.sel_]


GBM_I = CLASSES.index(GBM)
LGG_I = [CLASSES.index(c) for c in LGG]


class Hierarchical:
    """Stage 1: GBM vs LGG from the 4-class modality probabilities.
    Stage 2: astro / mixed / oligo from the LGG-specific modality models (trained on LGG patients only), pooled by
    a combiner fitted on LGG training patients only. Decision: GBM if stage-1 P(GBM) >= 0.5, else the stage-2 argmax."""
    def __init__(self, pool="learned"):
        self.pool = pool

    @staticmethod
    def _stage2(P, M):
        return {m: P["lgg:" + m] for m in MODALITIES}, {m: M["lgg:" + m] for m in MODALITIES}

    def fit(self, y, P, M):
        self.s1 = LearnedWeights() if self.pool == "learned" else Mean()
        self.s1.fit(y, P, M)
        lg = y != GBM_I
        P2, M2 = self._stage2(sub(P, lg), sub(M, lg))
        y2 = np.array([LGG_I.index(v) for v in y[lg]])
        self.s2 = LearnedWeights(k=3) if self.pool == "learned" else Mean()
        self.s2.fit(y2, P2, M2)
        return self

    def proba(self, P, M):
        pg = self.s1.proba(P, M)[:, GBM_I]
        P2, M2 = self._stage2(P, M)
        p2 = self.s2.proba(P2, M2)
        out = np.zeros((len(pg), K))
        out[:, LGG_I] = (1 - pg)[:, None] * p2
        out[:, GBM_I] = np.where(pg >= 0.5, 1.0, 0.0)      # makes argmax follow the two-stage rule
        return out


def method_factory(fam="LR"):
    meths = {f"{SHORT_M[m]} only": (lambda m=m: Single(m)) for m in MODALITIES}
    meths.update({
        "Mean": Mean,
        "Reliability-weighted": Reliability,
        "Learned weights (global)": lambda: LearnedWeights(per_class=False),
        "Learned weights (per class)": lambda: LearnedWeights(per_class=True),
        "Stacking LR": StackLR,
        "Meta-LR (same 8 inputs as meta-VQC)": lambda: MetaVQC(classical=True),
    })
    if RUN_META_VQC and fam != "LRm":
        meths["Meta-VQC (8 qubits)"] = MetaVQC
    if RUN_HIERARCHICAL and fam == "LR":
        meths["Hierarchical (learned weights)"] = lambda: Hierarchical("learned")
        meths["Hierarchical (mean)"] = lambda: Hierarchical("mean")
    return meths

In [9]:
# ------------------------------------------------------------
# Layer-2 cross-validation on the training OOF table (same site folds)
# ------------------------------------------------------------
y_tr = np.array([CLASSES.index(c) for c in cohort.loc[TRAIN, "subtype"]])
fold_tr = FOLD.loc[TRAIN].values
META_CV, META_CV_PRED = {}, {}
for fam in BASE_FAMILIES:
    Ptr, Mtr = meta_arrays(OOF[fam], TRAIN, lgg_for(fam))
    for name, make in method_factory(fam).items():
        t0 = time.time()
        pred = np.zeros((len(TRAIN), K))
        for k in range(OUTER_FOLDS):
            a, b = np.where(fold_tr != k)[0], np.where(fold_tr == k)[0]
            pred[b] = make().fit(y_tr[a], sub(Ptr, a), sub(Mtr, a)).proba(sub(Ptr, b), sub(Mtr, b))
        META_CV[(fam, name)] = pred
        f = macro(y_tr, pred.argmax(1), labels=range(K))
        dt = time.time() - t0
        if dt > 5 or not name.endswith("only"):
            print(f"[{fam} base] {name:30s} CV macro-F1 {f:.3f}  ({dt:.0f}s)")

cv_rows = []
lg_tr = y_tr != GBM_I
for (fam, name), pred in META_CV.items():
    pr = pred.argmax(1)
    folds = [macro(y_tr[fold_tr == k], pr[fold_tr == k], labels=range(K)) for k in range(OUTER_FOLDS)]
    cv_rows.append({"base": fam, "method": name, "cv_macro_f1": macro(y_tr, pr, labels=range(K)),
                    "cv_fold_sd": np.std(folds, ddof=1),
                    "cv_lgg_only_macro_f1": macro(y_tr[lg_tr], pr[lg_tr], labels=LGG_I),
                    "cv_gbm_vs_lgg_bal_acc": balanced_accuracy_score(y_tr == GBM_I, pr == GBM_I),
                    **{f"cv_f1_{s}": v for s, v in zip(SHORT, f1_score(y_tr, pr, labels=range(K), average=None,
                                                                       zero_division=0))}})
CV_TABLE = pd.DataFrame(cv_rows).sort_values("cv_macro_f1", ascending=False).reset_index(drop=True)
# best single modality = best per-modality CV on its own patients (no fallback involved)
mod_cv = MOD_TABLE[MOD_TABLE["eval"] == "CV"].sort_values("macro_f1", ascending=False).iloc[0]
BEST_SINGLE = (mod_cv["base"], f"{mod_cv['modality']} only")
single = CV_TABLE[(CV_TABLE["base"] == BEST_SINGLE[0]) & (CV_TABLE["method"] == BEST_SINGLE[1])]
fusion = CV_TABLE[~CV_TABLE["method"].str.endswith(" only")]
CV_ARGMAX = tuple(fusion.iloc[0][["base", "method"]])
BEST_FUSION = tuple(PRIMARY_MODEL) if PRIMARY_MODEL else CV_ARGMAX
assert any((CV_TABLE["base"] == BEST_FUSION[0]) & (CV_TABLE["method"] == BEST_FUSION[1])), f"unknown model {BEST_FUSION}"
FINAL_LABEL = "primary model (named for interpretability)" if PRIMARY_MODEL else "final model (CV-argmax)"
_cv = lambda k: CV_TABLE.set_index(["base", "method"]).loc[k, "cv_macro_f1"]
print(f"\nBest single modality (CV on its own patients): {BEST_SINGLE} {mod_cv['macro_f1']:.3f}")
print(f"Highest-CV fusion method: {CV_ARGMAX} {_cv(CV_ARGMAX):.3f}")
print(f"Reported {FINAL_LABEL}: {BEST_FUSION} {_cv(BEST_FUSION):.3f}")
top_gap = fusion["cv_macro_f1"].iloc[0] - fusion["cv_macro_f1"].iloc[min(4, len(fusion) - 1)]
print(f"Top-5 fusion methods span {top_gap:.3f} in CV vs median fold SD {fusion['cv_fold_sd'].median():.3f}"
      + (" -> within noise; the ranking among them is not meaningful." if top_gap < fusion['cv_fold_sd'].median() else "."))
print("Note: the winner's CV score is slightly optimistic because it was picked from many methods; the held-out-site"
      " test score below is the unbiased estimate. Fold SD shows how much CV varies between site folds.")
display(CV_TABLE.round(3))
CV_TABLE.to_csv(os.path.join(OUTPUT_DIR, "fusion_cv_table.csv"), index=False)

[LR base] Mean                           CV macro-F1 0.634  (0s)
[LR base] Reliability-weighted           CV macro-F1 0.647  (0s)
[LR base] Learned weights (global)       CV macro-F1 0.653  (0s)
[LR base] Learned weights (per class)    CV macro-F1 0.651  (1s)
[LR base] Stacking LR                    CV macro-F1 0.639  (0s)
[LR base] Meta-LR (same 8 inputs as meta-VQC) CV macro-F1 0.660  (0s)
[LR base] Meta-VQC (8 qubits)            CV macro-F1 0.633  (291s)
[LR base] Hierarchical (learned weights) CV macro-F1 0.638  (0s)
[LR base] Hierarchical (mean)            CV macro-F1 0.658  (0s)
[VQC base] Mean                           CV macro-F1 0.632  (0s)
[VQC base] Reliability-weighted           CV macro-F1 0.633  (0s)
[VQC base] Learned weights (global)       CV macro-F1 0.634  (0s)
[VQC base] Learned weights (per class)    CV macro-F1 0.629  (1s)
[VQC base] Stacking LR                    CV macro-F1 0.594  (0s)
[VQC base] Meta-LR (same 8 inputs as meta-VQC) CV macro-F1 0.543  (0s)
[VQC ba

,base,method,cv_macro_f1,cv_fold_sd,cv_lgg_only_macro_f1,cv_gbm_vs_lgg_bal_acc,cv_f1_Astro,cv_f1_GBM,cv_f1_Mixed,cv_f1_Oligo
0,LR,miRNA only,0.671,0.116,0.571,0.984,0.656,0.978,0.538,0.512
1,LR,Meta-LR (same 8 inputs as meta-VQC),0.660,0.107,0.561,0.971,0.670,0.971,0.444,0.556
2,LR,Hierarchical (mean),0.658,0.088,0.562,0.964,0.670,0.963,0.434,0.567
3,LR,Learned weights (global),0.653,0.121,0.548,0.979,0.663,0.978,0.442,0.528
4,LR,Learned weights (per class),0.651,0.115,0.549,0.971,0.663,0.971,0.451,0.519
5,LR,Reliability-weighted,0.647,0.099,0.539,0.984,0.680,0.978,0.430,0.500
6,LR,Stacking LR,0.639,0.093,0.529,0.984,0.642,0.978,0.474,0.462
7,LR,Hierarchical (learned weights),0.638,0.113,0.531,0.971,0.632,0.971,0.427,0.520
8,VQC,Learned weights (global),0.634,0.117,0.527,0.971,0.635,0.971,0.431,0.500
9,LR,Mean,0.634,0.132,0.530,0.976,0.653,0.951,0.439,0.492


## 8. Held-out sites — every model scored once
Meta-learners are refitted on all training OOF probabilities and applied to the test probabilities of the final
per-modality models. **All test patients** use whatever modalities they have; **complete case** = the test patients
with all six. "<X> only" rows on all test patients fall back to the mean of the other modalities when X is missing
(so every row is scored on the same patients); `test_own_patients_macro_f1` scores a single modality only on the
test patients that have it (`n_own`), which is the clean single-modality number. `n_gbm_<platform>` gives the
number of test GBMs behind each platform-specific recall — with n ≤ 3 that recall is anecdotal.

In [10]:
def boot_ci(y, pred, n=N_BOOT, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); y, pred = np.asarray(y), np.asarray(pred)
    v = [macro(y[i], pred[i], labels=range(K)) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(v, [2.5, 97.5])


def paired_boot(y, pa, pb, n=N_BOOT, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); y, pa, pb = map(np.asarray, (y, pa, pb))
    d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        d.append(macro(y[i], pa[i], labels=range(K)) - macro(y[i], pb[i], labels=range(K)))
    d = np.array(d); return d.mean(), np.percentile(d, [2.5, 97.5])


y_te = np.array([CLASSES.index(c) for c in cohort.loc[TEST, "subtype"]])
cc_te = cohort.loc[TEST, "complete"].values
plat_te = cohort.loc[TEST, "meth_platform"].values
FITTED, TEST_PRED, rows = {}, {}, []
MOD_OF = {f"{SHORT_M[m]} only": m for m in MODALITIES}
TEST_MASKS = {}
for fam in BASE_FAMILIES:
    Ptr, Mtr = meta_arrays(OOF[fam], TRAIN, lgg_for(fam))
    Pte, Mte = meta_arrays(TESTP[fam], TEST, lgg_test_for(fam))
    TEST_MASKS[fam] = Mte
    for name, make in method_factory(fam).items():
        mdl = make().fit(y_tr, Ptr, Mtr)
        FITTED[(fam, name)] = mdl
        pr = mdl.proba(Pte, Mte).argmax(1)
        TEST_PRED[(fam, name)] = pr
        lo, hi = boot_ci(y_te, pr)
        f = f1_score(y_te, pr, labels=range(K), average=None, zero_division=0)
        lg = y_te != CLASSES.index(GBM)
        g = y_te == CLASSES.index(GBM)
        own = Mte[MOD_OF[name]] if name in MOD_OF else np.ones(len(y_te), bool)
        rows.append({
            "base": fam, "method": name,
            "cv_macro_f1": macro(y_tr, META_CV[(fam, name)].argmax(1), labels=range(K)),
            "test_macro_f1": macro(y_te, pr, labels=range(K)), "ci_low": lo, "ci_high": hi,
            "test_complete_case_macro_f1": macro(y_te[cc_te], pr[cc_te], labels=range(K)),
            "test_own_patients_macro_f1": macro(y_te[own], pr[own], labels=range(K)), "n_own": int(own.sum()),
            "test_balanced_acc": balanced_accuracy_score(y_te, pr),
            "test_lgg_only_macro_f1": macro(y_te[lg], pr[lg], labels=[CLASSES.index(c) for c in LGG]),
            **{f"test_f1_{s}": v for s, v in zip(SHORT, f)},
            **{f"gbm_recall_{p}": (float((pr[g & (plat_te == p)] == CLASSES.index(GBM)).mean())
                                   if (g & (plat_te == p)).sum() else np.nan) for p in ["27K", "450K", "none"]},
            **{f"n_gbm_{p}": int((g & (plat_te == p)).sum()) for p in ["27K", "450K", "none"]},
        })
RESULTS = pd.DataFrame(rows).sort_values(["base", "cv_macro_f1"], ascending=[True, False]).reset_index(drop=True)
RESULTS = RESULTS.merge(CV_TABLE.drop(columns=["cv_macro_f1"]), on=["base", "method"], how="left")
display(RESULTS.round(3))
print("Test patients:", len(TEST), "| complete case:", int(cc_te.sum()),
      "| test GBM by methylation platform:", pd.Series(plat_te[y_te == CLASSES.index(GBM)]).value_counts().to_dict())
RESULTS.to_csv(os.path.join(OUTPUT_DIR, "fusion_results_all_methods.csv"), index=False)

,base,method,cv_macro_f1,test_macro_f1,ci_low,ci_high,test_complete_case_macro_f1,test_own_patients_macro_f1,n_own,test_balanced_acc,test_lgg_only_macro_f1,test_f1_Astro,test_f1_GBM,test_f1_Mixed,test_f1_Oligo,gbm_recall_27K,gbm_recall_450K,gbm_recall_none,n_gbm_27K,n_gbm_450K,n_gbm_none,cv_fold_sd,cv_lgg_only_macro_f1,cv_gbm_vs_lgg_bal_acc,cv_f1_Astro,cv_f1_GBM,cv_f1_Mixed,cv_f1_Oligo
0,LR,miRNA only,0.671,0.694,0.612,0.765,0.674,0.694,111,0.710,0.592,0.667,1.000,0.508,0.600,1.000,1.0,1.0,27,2,1,0.116,0.571,0.984,0.656,0.978,0.538,0.512
1,LR,Meta-LR (same 8 inputs as meta-VQC),0.660,0.640,0.560,0.708,0.657,0.640,124,0.676,0.519,0.598,1.000,0.308,0.653,1.000,1.0,1.0,27,2,1,0.107,0.561,0.971,0.670,0.971,0.444,0.556
2,LR,Hierarchical (mean),0.658,0.726,0.650,0.797,0.714,0.726,124,0.736,0.634,0.659,1.000,0.500,0.744,1.000,1.0,1.0,27,2,1,0.088,0.562,0.964,0.670,0.963,0.434,0.567
3,LR,Learned weights (global),0.653,0.706,0.628,0.778,0.708,0.706,124,0.713,0.608,0.613,1.000,0.500,0.711,1.000,1.0,1.0,27,2,1,0.121,0.548,0.979,0.663,0.978,0.442,0.528
4,LR,Learned weights (per class),0.651,0.698,0.619,0.770,0.701,0.698,124,0.705,0.597,0.603,1.000,0.493,0.696,1.000,1.0,1.0,27,2,1,0.115,0.549,0.971,0.663,0.971,0.451,0.519
5,LR,Reliability-weighted,0.647,0.677,0.593,0.748,0.603,0.677,124,0.693,0.590,0.597,0.938,0.444,0.727,1.000,1.0,1.0,27,2,1,0.099,0.539,0.984,0.680,0.978,0.430,0.500
6,LR,Stacking LR,0.639,0.627,0.554,0.693,0.637,0.627,124,0.674,0.503,0.638,1.000,0.204,0.667,1.000,1.0,1.0,27,2,1,0.093,0.529,0.984,0.642,0.978,0.474,0.462
7,LR,Hierarchical (learned weights),0.638,0.689,0.610,0.759,0.691,0.689,124,0.700,0.585,0.590,1.000,0.469,0.696,1.000,1.0,1.0,27,2,1,0.113,0.531,0.971,0.632,0.971,0.427,0.520
8,LR,Mean,0.634,0.672,0.587,0.746,0.571,0.672,124,0.693,0.597,0.605,0.896,0.459,0.727,1.000,1.0,1.0,27,2,1,0.132,0.530,0.976,0.653,0.951,0.439,0.492
9,LR,Meta-VQC (8 qubits),0.633,0.684,0.601,0.756,0.699,0.684,124,0.692,0.579,0.617,1.000,0.469,0.651,1.000,1.0,1.0,27,2,1,0.121,0.542,0.949,0.626,0.928,0.467,0.513


Test patients: 124 | complete case: 81 | test GBM by methylation platform: {'27K': 27, '450K': 2, 'none': 1}


In [11]:
print(f"Paired bootstrap on the test set: each fusion method minus the best single modality {BEST_SINGLE}")
PAIRED = {}
for key in FITTED:
    if key[1].endswith(" only") or key == BEST_SINGLE:
        continue
    d, ci = paired_boot(y_te, TEST_PRED[key], TEST_PRED[BEST_SINGLE])
    PAIRED[key] = (d, ci)
    flag = "better" if ci[0] > 0 else ("worse" if ci[1] < 0 else "not distinguishable")
    star = (f"  <- {FINAL_LABEL}" if key == BEST_FUSION else "") + ("  <- highest CV" if key == CV_ARGMAX and key != BEST_FUSION else "")
    if key == BEST_FUSION:
        dc, cic = paired_boot(y_te[cc_te], TEST_PRED[key][cc_te], TEST_PRED[BEST_SINGLE][cc_te])
        PAIRED_CC = (dc, cic)
        own = TEST_MASKS[BEST_SINGLE[0]][MOD_OF[BEST_SINGLE[1]]]
        do, cio = paired_boot(y_te[own], TEST_PRED[key][own], TEST_PRED[BEST_SINGLE][own])
        PAIRED_OWN = (do, cio, int(own.sum()))
        star += (f" | complete-case ΔF1 {dc:+.3f} [{cic[0]:+.3f}, {cic[1]:+.3f}]"
                 f" | patients with {BEST_SINGLE[1].replace(' only', '')} ΔF1 {do:+.3f} [{cio[0]:+.3f}, {cio[1]:+.3f}]")
    print(f"  [{key[0]}] {key[1]:30s} ΔF1 {d:+.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]  {flag}{star}")

if RUN_BASE_VQC:
    print("\nQuantum vs classical, same fusion method, test macro-F1 (VQC base minus LR base):")
    for name in method_factory("LR"):
        if ("VQC", name) in TEST_PRED and not name.endswith(" only"):
            d, ci = paired_boot(y_te, TEST_PRED[("VQC", name)], TEST_PRED[("LR", name)])
            print(f"  {name:30s} {d:+.3f}  [{ci[0]:+.3f}, {ci[1]:+.3f}]")
if RUN_META_VQC:
    for fam in [f for f in BASE_FAMILIES if (f, "Meta-VQC (8 qubits)") in TEST_PRED]:
        d, ci = paired_boot(y_te, TEST_PRED[(fam, "Meta-VQC (8 qubits)")], TEST_PRED[(fam, "Meta-LR (same 8 inputs as meta-VQC)")])
        print(f"\n[{fam} base] Meta-VQC minus LR on the same 8 inputs: {d:+.3f}  [{ci[0]:+.3f}, {ci[1]:+.3f}]")

Paired bootstrap on the test set: each fusion method minus the best single modality ('LR', 'miRNA only')
  [LR] Mean                           ΔF1 -0.023  95% CI [-0.105, +0.056]  not distinguishable
  [LR] Reliability-weighted           ΔF1 -0.018  95% CI [-0.093, +0.058]  not distinguishable
  [LR] Learned weights (global)       ΔF1 +0.012  95% CI [-0.059, +0.083]  not distinguishable
  [LR] Learned weights (per class)    ΔF1 +0.004  95% CI [-0.069, +0.074]  not distinguishable
  [LR] Stacking LR                    ΔF1 -0.067  95% CI [-0.148, +0.022]  not distinguishable
  [LR] Meta-LR (same 8 inputs as meta-VQC) ΔF1 -0.054  95% CI [-0.129, +0.026]  not distinguishable  <- highest CV
  [LR] Meta-VQC (8 qubits)            ΔF1 -0.010  95% CI [-0.081, +0.062]  not distinguishable
  [LR] Hierarchical (learned weights) ΔF1 -0.005  95% CI [-0.080, +0.071]  not distinguishable
  [LR] Hierarchical (mean)            ΔF1 +0.032  95% CI [-0.044, +0.109]  not distinguishable
  [VQC] Mean        

In [12]:
# ------------------------------------------------------------
# Final model: report + confusion matrix
# ------------------------------------------------------------
pf = TEST_PRED[BEST_FUSION]
print(f"Reported {FINAL_LABEL}: {BEST_FUSION}\n")
print(classification_report(y_te, pf, labels=range(K), target_names=CLASSES, digits=3, zero_division=0))
display(pd.DataFrame(confusion_matrix(y_te, pf, labels=range(K)), index=[f"true_{s}" for s in SHORT],
                     columns=[f"pred_{s}" for s in SHORT]))
_cm = confusion_matrix(y_te, pf, labels=range(K)); _err = _cm.sum() - np.trace(_cm)
_pairs = {f"{SHORT[i]}<->{SHORT[j]}": int(_cm[i, j] + _cm[j, i]) for i in range(K) for j in range(i + 1, K)
          if _cm[i, j] + _cm[j, i] > 0}
print(f"Where the {_err} errors are: " + ", ".join(f"{k} {v} ({v / max(_err, 1):.0%})"
      for k, v in sorted(_pairs.items(), key=lambda t: -t[1])))
lg = y_te != CLASSES.index(GBM)
print("LGG-only (least site/platform-confounded comparison):")
print(classification_report(y_te[lg], pf[lg], labels=[CLASSES.index(c) for c in LGG], target_names=LGG,
                            digits=3, zero_division=0))
if isinstance(FITTED[BEST_FUSION], MetaVQC):
    print("Meta-VQC inputs (qubit order):", FITTED[BEST_FUSION].feature_names())
pd.DataFrame({"patient": TEST, "true": cohort.loc[TEST, "subtype"].values, "pred": np.array(CLASSES)[pf],
              "site": cohort.loc[TEST, "site"].values, "n_modalities": cohort.loc[TEST, "n_modalities"].values,
              "meth_platform": plat_te}).to_csv(os.path.join(OUTPUT_DIR, "final_model_test_predictions.csv"), index=False)

Reported primary model (named for interpretability): ('VQC', 'Learned weights (global)')

                   precision    recall  f1-score   support

      astrocytoma      0.600     0.706     0.649        34
     glioblastoma      0.968     1.000     0.984        30
     mixed_glioma      0.593     0.421     0.492        38
oligodendroglioma      0.615     0.727     0.667        22

         accuracy                          0.694       124
        macro avg      0.694     0.714     0.698       124
     weighted avg      0.689     0.694     0.685       124



,pred_Astro,pred_GBM,pred_Mixed,pred_Oligo
true_Astro,24,0,8,2
true_GBM,0,30,0,0
true_Mixed,13,1,16,8
true_Oligo,3,0,3,16


Where the 38 errors are: Astro<->Mixed 21 (55%), Mixed<->Oligo 11 (29%), Astro<->Oligo 5 (13%), GBM<->Mixed 1 (3%)
LGG-only (least site/platform-confounded comparison):
                   precision    recall  f1-score   support

      astrocytoma      0.600     0.706     0.649        34
     mixed_glioma      0.593     0.421     0.492        38
oligodendroglioma      0.615     0.727     0.667        22

        micro avg      0.602     0.596     0.599        94
        macro avg      0.603     0.618     0.603        94
     weighted avg      0.601     0.596     0.590        94



## 9. Modality weights and leave-one-modality-out importance
Learned weights show how much the log-linear pool trusts each modality (fitted on all training OOF probabilities).
The ablation removes one modality at a time (masked for every patient) and re-runs the fusion CV — the drop in CV
macro-F1 is that modality's contribution *given the other five*.

In [13]:
for fam in BASE_FAMILIES:
    w = FITTED[(fam, "Learned weights (global)")].weights()
    print(f"[{fam} base] learned global modality weights. Absolute values < 1 mean the six-way product was"
          " over-confident and is tempered; compare modalities by their relative share:")
    display(pd.DataFrame({"weight": w, "relative_share": w / w.sum()}).round(3))
    print(f"[{fam} base] learned per-class weights:")
    display(FITTED[(fam, "Learned weights (per class)")].weights().round(2))

fam_ab = BEST_FUSION[0]
ab_methods = {"Learned weights (global)": lambda: LearnedWeights(per_class=False), "Stacking LR": StackLR}
if BEST_FUSION[1] not in ab_methods and not BEST_FUSION[1].startswith("Meta-VQC"):
    ab_methods[BEST_FUSION[1]] = method_factory(BEST_FUSION[0])[BEST_FUSION[1]]
Ptr, Mtr = meta_arrays(OOF[fam_ab], TRAIN, lgg_for(fam_ab))
ab_rows = []
for name, make in ab_methods.items():
    def cv_with(Mx):
        pred = np.zeros((len(TRAIN), K))
        for k in range(OUTER_FOLDS):
            a, b = np.where(fold_tr != k)[0], np.where(fold_tr == k)[0]
            pred[b] = make().fit(y_tr[a], sub(Ptr, a), sub(Mx, a)).proba(sub(Ptr, b), sub(Mx, b))
        return macro(y_tr, pred.argmax(1), labels=range(K))
    full = cv_with(Mtr)
    for m in MODALITIES:
        Mx = dict(Mtr); Mx[m] = np.zeros_like(Mtr[m])
        if "lgg:" + m in Mx:
            Mx["lgg:" + m] = np.zeros_like(Mtr[m])
        ab_rows.append({"method": name, "removed": SHORT_M[m], "cv_macro_f1": cv_with(Mx), "full": full})
AB = pd.DataFrame(ab_rows); AB["drop_when_removed"] = AB["full"] - AB["cv_macro_f1"]
print(f"\nLeave-one-modality-out ({fam_ab} base): larger drop = more important")
AB_PIV = AB.pivot(index="removed", columns="method", values="drop_when_removed").sort_values(
    "Learned weights (global)", ascending=False)
display(AB_PIV.round(3))
print("A drop near 0 means the other modalities already carry that information (redundancy), not that the modality"
      " is useless on its own — compare with its single-modality score in section 6.")
AB.to_csv(os.path.join(OUTPUT_DIR, "modality_ablation.csv"), index=False)

[LR base] learned global modality weights. Absolute values < 1 mean the six-way product was over-confident and is tempered; compare modalities by their relative share:


,weight,relative_share
PROT,0.555,0.175
METH,0.425,0.134
GE,0.532,0.168
miRNA,0.553,0.175
SNV,0.477,0.151
CNV,0.626,0.197


[LR base] learned per-class weights:


,Astro,GBM,Mixed,Oligo
PROT,0.91,0.96,0.82,0.81
METH,0.88,1.00,0.73,0.67
GE,0.83,1.03,0.80,0.78
miRNA,0.89,1.07,0.80,0.72
SNV,1.00,0.96,0.72,0.74
CNV,0.97,0.98,0.84,0.85


[VQC base] learned global modality weights. Absolute values < 1 mean the six-way product was over-confident and is tempered; compare modalities by their relative share:


,weight,relative_share
PROT,0.885,0.252
METH,0.550,0.157
GE,0.465,0.133
miRNA,0.634,0.181
SNV,0.436,0.124
CNV,0.538,0.153


[VQC base] learned per-class weights:


,Astro,GBM,Mixed,Oligo
PROT,0.99,0.98,1.02,0.97
METH,0.88,0.97,0.88,0.77
GE,0.81,1.04,0.76,0.67
miRNA,0.89,1.14,0.83,0.73
SNV,0.91,0.95,0.69,0.72
CNV,0.96,0.94,0.82,0.77



Leave-one-modality-out (VQC base): larger drop = more important


method,Learned weights (global),Stacking LR
removed,,
miRNA,0.125,0.087
CNV,0.017,0.013
GE,0.013,-0.002
METH,-0.002,-0.008
PROT,-0.013,-0.041
SNV,-0.015,0.014


A drop near 0 means the other modalities already carry that information (redundancy), not that the modality is useless on its own — compare with its single-modality score in section 6.


## 10. Summary

In [14]:
summary = RESULTS[["base", "method", "cv_macro_f1", "test_macro_f1", "ci_low", "ci_high",
                   "test_complete_case_macro_f1", "test_lgg_only_macro_f1",
                   "gbm_recall_27K", "gbm_recall_450K"]].copy()
summary["final"] = [(b, m) == BEST_FUSION for b, m in zip(summary["base"], summary["method"])]
display(summary.sort_values("cv_macro_f1", ascending=False).round(3).reset_index(drop=True))
summary.to_csv(os.path.join(OUTPUT_DIR, "fusion_summary.csv"), index=False)

r = RESULTS.set_index(["base", "method"])
fin, sgl = r.loc[BEST_FUSION], r.loc[BEST_SINGLE]
print(f"Cohort: {len(cohort)} patients (train {len(TRAIN)}, test {len(TEST)} from {sorted(set(cohort.loc[TEST, 'site']))}); "
      f"complete case: train {int(cohort.loc[TRAIN, 'complete'].sum())}, test {int(cc_te.sum())}")
print(f"Reported {FINAL_LABEL}: {BEST_FUSION}: CV {fin['cv_macro_f1']:.3f} | TEST {fin['test_macro_f1']:.3f} "
      f"(95% CI {fin['ci_low']:.3f}–{fin['ci_high']:.3f}) | complete-case {fin['test_complete_case_macro_f1']:.3f} "
      f"| LGG-only {fin['test_lgg_only_macro_f1']:.3f}")
print(f"Best single modality (chosen by CV): {BEST_SINGLE}: CV {sgl['cv_macro_f1']:.3f} | TEST {sgl['test_macro_f1']:.3f}")
d, ci = PAIRED[BEST_FUSION]
verdict = ("fusion significantly better" if ci[0] > 0 else
           "fusion significantly worse" if ci[1] < 0 else "difference not distinguishable on this test set")
print(f"Final fusion minus best single modality on test: {d:+.3f} (95% CI {ci[0]:+.3f} to {ci[1]:+.3f}) -> {verdict}")
print(f"  ... on the complete-case test patients: {PAIRED_CC[0]:+.3f} (95% CI {PAIRED_CC[1][0]:+.3f} to {PAIRED_CC[1][1]:+.3f})")
if CV_ARGMAX != BEST_FUSION:
    a = r.loc[CV_ARGMAX]
    print(f"Highest-CV fusion method, reported alongside: {CV_ARGMAX}: CV {a['cv_macro_f1']:.3f} | TEST {a['test_macro_f1']:.3f} "
          f"(95% CI {a['ci_low']:.3f}–{a['ci_high']:.3f})")
    print("Disclosure: the primary model was named for interpretability after the top CV methods were found to be"
          " within fold noise; the CV-argmax result above is the strictly pre-declared alternative.")
print("Reminder: GBM metrics are site- and (for methylation) platform-confounded; LGG-only and complete-case columns"
      " are the cleaner comparisons. Choose models by the CV column, never by the test column.")
json.dump({"final_model": list(BEST_FUSION), "cv_argmax": list(CV_ARGMAX), "best_single": list(BEST_SINGLE), "base_models": BASE_INFO},
          open(os.path.join(OUTPUT_DIR, "fusion_config.json"), "w"), indent=1, default=str)

,base,method,cv_macro_f1,test_macro_f1,ci_low,ci_high,test_complete_case_macro_f1,test_lgg_only_macro_f1,gbm_recall_27K,gbm_recall_450K,final
0,LR,miRNA only,0.671,0.694,0.612,0.765,0.674,0.592,1.000,1.0,False
1,LR,Meta-LR (same 8 inputs as meta-VQC),0.660,0.640,0.560,0.708,0.657,0.519,1.000,1.0,False
2,LR,Hierarchical (mean),0.658,0.726,0.650,0.797,0.714,0.634,1.000,1.0,False
3,LR,Learned weights (global),0.653,0.706,0.628,0.778,0.708,0.608,1.000,1.0,False
4,LR,Learned weights (per class),0.651,0.698,0.619,0.770,0.701,0.597,1.000,1.0,False
5,LR,Reliability-weighted,0.647,0.677,0.593,0.748,0.603,0.590,1.000,1.0,False
6,LR,Stacking LR,0.639,0.627,0.554,0.693,0.637,0.503,1.000,1.0,False
7,LR,Hierarchical (learned weights),0.638,0.689,0.610,0.759,0.691,0.585,1.000,1.0,False
8,VQC,Learned weights (global),0.634,0.698,0.618,0.769,0.713,0.603,1.000,1.0,True
9,LR,Mean,0.634,0.672,0.587,0.746,0.571,0.597,1.000,1.0,False


Cohort: 439 patients (train 315, test 124 from ['12', '14', '27', 'CS', 'DU', 'QH', 'RY']); complete case: train 208, test 81
Reported primary model (named for interpretability): ('VQC', 'Learned weights (global)'): CV 0.634 | TEST 0.698 (95% CI 0.618–0.769) | complete-case 0.713 | LGG-only 0.603
Best single modality (chosen by CV): ('LR', 'miRNA only'): CV 0.671 | TEST 0.694
Final fusion minus best single modality on test: +0.004 (95% CI -0.076 to +0.080) -> difference not distinguishable on this test set
  ... on the complete-case test patients: +0.040 (95% CI -0.048 to +0.124)
Highest-CV fusion method, reported alongside: ('LR', 'Meta-LR (same 8 inputs as meta-VQC)'): CV 0.660 | TEST 0.640 (95% CI 0.560–0.708)
Disclosure: the primary model was named for interpretability after the top CV methods were found to be within fold noise; the CV-argmax result above is the strictly pre-declared alternative.
Reminder: GBM metrics are site- and (for methylation) platform-confounded; LGG-only an

## 11. Questions a reviewer will ask — answered from this run
Every answer below is computed from the tables above; nothing is typed in by hand.

In [15]:
R = RESULTS.set_index(["base", "method"])
def line(q, a):
    print(f"Q: {q}\nA: {a}\n")

fin, sgl = R.loc[BEST_FUSION], R.loc[BEST_SINGLE]
d, ci = PAIRED[BEST_FUSION]
_a = R.loc[CV_ARGMAX]
line("How was the final model chosen, and could the test set have influenced it?",
     (f"{BEST_FUSION[1]} ({BEST_FUSION[0]} base) is the primary model, named for interpretability (one weight per "
      f"modality). The top fusion methods were within CV noise (spread {top_gap:.3f} vs fold SD "
      f"{fusion['cv_fold_sd'].median():.3f}). The highest-CV method, {CV_ARGMAX[1]} ({CV_ARGMAX[0]}), is reported "
      f"alongside: CV {_a['cv_macro_f1']:.3f}, test {_a['test_macro_f1']:.3f} ({_a['ci_low']:.3f}-{_a['ci_high']:.3f}). "
      "This naming was made after an earlier run, so it is disclosed; all methods are reported.")
     if CV_ARGMAX != BEST_FUSION else
     f"Pre-declared rule: highest CV macro-F1 on training sites among all fusion methods -> {BEST_FUSION[1]} "
     f"({BEST_FUSION[0]} base models). The test set was only used after this choice; all methods are reported.")
line("How well does the final model do on unseen hospitals?",
     f"Test macro-F1 {fin['test_macro_f1']:.3f} (95% CI {fin['ci_low']:.3f}-{fin['ci_high']:.3f}, n={len(TEST)} patients "
     f"from {len(set(cohort.loc[TEST, 'site']))} held-out sites); CV {fin['cv_macro_f1']:.3f} +/- {fin['cv_fold_sd']:.3f} (fold SD).")
line("Does fusion beat the best single modality?",
     f"Best single modality by CV: {BEST_SINGLE[1]} ({BEST_SINGLE[0]}), CV {mod_cv['macro_f1']:.3f}. "
     f"Paired test difference (fusion - single): {d:+.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}] on all test patients; "
     f"{PAIRED_CC[0]:+.3f} [{PAIRED_CC[1][0]:+.3f}, {PAIRED_CC[1][1]:+.3f}] on complete-case patients; "
     f"{PAIRED_OWN[0]:+.3f} [{PAIRED_OWN[1][0]:+.3f}, {PAIRED_OWN[1][1]:+.3f}] on the {PAIRED_OWN[2]} test patients that have "
     f"{BEST_SINGLE[1].replace(' only', '')}. " + ("Fusion is significantly better." if ci[0] > 0 else
     "Fusion is significantly worse." if ci[1] < 0 else "No significant difference: report them as comparable."))
line("What does fusion add if it is not more accurate?",
     f"Coverage: it predicts all {len(TEST)} test patients from whatever data they have; the best single modality "
     f"exists for {PAIRED_OWN[2]} of them.")
top = AB_PIV["Learned weights (global)"].sort_values(ascending=False)
line("Which modalities matter?",
     "Leave-one-out CV drop (learned weights): " + ", ".join(f"{i} {v:+.3f}" for i, v in top.items()) +
     ". Single-modality CV (own patients, LR): " + ", ".join(
         f"{r.modality} {r.macro_f1:.3f}" for r in MOD_TABLE[(MOD_TABLE['eval'] == 'CV') & (MOD_TABLE['base'] == 'LR')]
         .sort_values('macro_f1', ascending=False).itertuples()) + ". Small drops with good single scores = redundancy.")
if RUN_HIERARCHICAL:
    h = R.loc[("LR", "Hierarchical (learned weights)")]; lw = R.loc[("LR", "Learned weights (global)")]
    line("Does a dedicated lower-grade-glioma stage help astro / mixed / oligo?",
         f"CV LGG-only macro-F1: hierarchical {h['cv_lgg_only_macro_f1']:.3f} vs flat learned weights "
         f"{lw['cv_lgg_only_macro_f1']:.3f}; test LGG-only {h['test_lgg_only_macro_f1']:.3f} vs {lw['test_lgg_only_macro_f1']:.3f}.")
if RUN_BASE_VQC or RUN_META_VQC:
    parts = []
    if RUN_BASE_VQC:
        for name in ["Learned weights (global)", "Stacking LR"]:
            dq, ciq = paired_boot(y_te, TEST_PRED[("VQC", name)], TEST_PRED[("LR", name)])
            parts.append(f"VQC vs LR base models under '{name}': {dq:+.3f} [{ciq[0]:+.3f}, {ciq[1]:+.3f}]")
    if RUN_META_VQC:
        for fam in [f for f in BASE_FAMILIES if (f, "Meta-VQC (8 qubits)") in TEST_PRED]:
            dq, ciq = paired_boot(y_te, TEST_PRED[(fam, "Meta-VQC (8 qubits)")], TEST_PRED[(fam, "Meta-LR (same 8 inputs as meta-VQC)")])
            parts.append(f"meta-VQC vs LR on the same 8 inputs ({fam} base): {dq:+.3f} [{ciq[0]:+.3f}, {ciq[1]:+.3f}]")
    line("Does the quantum model help?", "; ".join(parts) +
         ". Intervals containing 0 = no demonstrated quantum advantage (all circuits are exact noiseless simulations).")
line("Is the GBM performance biology or batch?",
     f"GBM sites contributed only GBM (site confound). Final model GBM recall: 27K-array {fin['gbm_recall_27K']:.2f} "
     f"(n={int(fin['n_gbm_27K'])}), 450K-array {fin['gbm_recall_450K']:.2f} (n={int(fin['n_gbm_450K'])}), no methylation "
     f"{fin['gbm_recall_none']:.2f} (n={int(fin['n_gbm_none'])}). The LGG-only score "
     f"(CV {fin['cv_lgg_only_macro_f1']:.3f}, test {fin['test_lgg_only_macro_f1']:.3f}) is the less confounded number.")
pcf = {c: fin[f"cv_f1_{c}"] for c in SHORT}
line("Which class limits 4-class performance?",
     "Final-model CV F1 per class: " + ", ".join(f"{c} {v:.3f}" for c, v in pcf.items()) +
     f". Lowest: {min(pcf, key=pcf.get)}. Mixed glioma has had no molecular definition since WHO 2016, so it is the"
     " expected ceiling; the LGG-only and per-class numbers show how much of the error it accounts for.")

Q: How was the final model chosen, and could the test set have influenced it?
A: Learned weights (global) (VQC base) is the primary model, named for interpretability (one weight per modality). The top fusion methods were within CV noise (spread 0.013 vs fold SD 0.114). The highest-CV method, Meta-LR (same 8 inputs as meta-VQC) (LR), is reported alongside: CV 0.660, test 0.640 (0.560-0.708). This naming was made after an earlier run, so it is disclosed; all methods are reported.

Q: How well does the final model do on unseen hospitals?
A: Test macro-F1 0.698 (95% CI 0.618-0.769, n=124 patients from 7 held-out sites); CV 0.634 +/- 0.117 (fold SD).

Q: Does fusion beat the best single modality?
A: Best single modality by CV: miRNA only (LR), CV 0.669. Paired test difference (fusion - single): +0.004 [-0.076, +0.080] on all test patients; +0.040 [-0.048, +0.124] on complete-case patients; +0.002 [-0.077, +0.084] on the 111 test patients that have miRNA. No significant difference: report th

In [16]:
# ============================================================
# EXPORT FOR PART A — paste as the LAST cell of fusion-final.ipynb and run the notebook once.
# Saves per-patient class probabilities of every fusion method (test patients + training-site CV),
# so the extra-metrics notebook can compute AUC, calibration, per-hospital scores and Holm-corrected tests.
# Nothing is refitted here: it only calls .proba() on the models already fitted above.
# ============================================================
rows = []
for fam in BASE_FAMILIES:
    Pte, Mte = meta_arrays(TESTP[fam], TEST, lgg_test_for(fam))
    for name in method_factory(fam):
        key = (fam, name)
        if key not in FITTED:
            continue
        for ids, P_, st in [(TEST, FITTED[key].proba(Pte, Mte), "test"), (TRAIN, META_CV[key], "cv")]:
            d = pd.DataFrame(np.asarray(P_, float), columns=CLASSES)
            d.insert(0, "set", st); d.insert(0, "method", name); d.insert(0, "base", fam)
            d.insert(0, "patient", [str(i) for i in ids])
            rows.append(d)
FUS_EXPORT = pd.concat(rows, ignore_index=True)
FUS_EXPORT.to_csv(os.path.join(OUTPUT_DIR, "fusion_probs_all_methods.csv"), index=False)
# sanity check: argmax of the exported probabilities must give the same test predictions as above
for key, pr in TEST_PRED.items():
    d = FUS_EXPORT[(FUS_EXPORT.base == key[0]) & (FUS_EXPORT.method == key[1]) & (FUS_EXPORT.set == "test")]
    assert (d[CLASSES].values.argmax(1) == pr).all(), f"mismatch for {key}"
print(f"Saved fusion_probs_all_methods.csv ({len(FUS_EXPORT):,} rows, {FUS_EXPORT[['base', 'method']].drop_duplicates().shape[0]} "
      f"methods) -> {OUTPUT_DIR}. Predictions match TEST_PRED for every method.")


Saved fusion_probs_all_methods.csv (12,292 rows, 28 methods) -> /kaggle/working. Predictions match TEST_PRED for every method.


In [17]:
# ============================================================
# 12. FUSION ON THE IBM HARDWARE OUTPUTS (no QPU time)
# Inputs: hw_probs_<modality>.csv from cell E6 of the six modality notebooks (attach them as one Kaggle dataset).
# The fusion models are the ones already fitted above on the training-site OOF VQC probabilities (FITTED) — nothing is
# refitted; only the test-patient inputs are swapped:
#   (a) simulator, full ensembles  = the result in the paper
#   (b) simulator, same models as on hardware (one ensemble member for methylation / miRNA / SNV) = matched reference
#   (c) IBM hardware outputs of those same models
# (c) vs (b) isolates the effect of the quantum hardware on the fused prediction.
# ============================================================
hw_paths = {m: sorted(glob.glob(f"/kaggle/input/**/hw_probs_{m}.csv", recursive=True)) for m in MODALITIES}
missing = [m for m, p in hw_paths.items() if not p]
assert not missing, f"hw_probs file(s) not found for {missing}: attach the dataset with the six hw_probs_<modality>.csv files"
HWP, NFP = {}, {}
for m, p in hw_paths.items():
    d = pd.read_csv(p[0]); d["patient"] = d["patient"].astype(str)
    HWP[m] = d[d.kind == "hardware"].set_index("patient")[CLASSES].astype(float)
    NFP[m] = d[d.kind == "noise_free_same_model"].set_index("patient")[CLASSES].astype(float)
    assert set(HWP[m].index) == set(TESTP["VQC"][m].index), f"{m}: hardware patients differ from the VQC test patients"
SCEN = {"(a) simulator, full ensembles": TESTP["VQC"], "(b) simulator, same models as hardware": NFP, "(c) IBM hardware": HWP}
HW_METHODS = ["Learned weights (global)", "Mean", "Reliability-weighted", "Learned weights (per class)", "Stacking LR",
              "Meta-VQC (8 qubits)"]
hw_rows, HW_PRED = [], {}
for name in HW_METHODS:
    mdl = FITTED.get(("VQC", name))
    if mdl is None:
        continue
    for sl, Pd in SCEN.items():
        Pte, Mte = meta_arrays(Pd, TEST)
        pr = mdl.proba(Pte, Mte).argmax(1); HW_PRED[(name, sl)] = pr
        lo, hi = boot_ci(y_te, pr); lg = y_te != CLASSES.index(GBM)
        r = {"fusion (VQC base)": name + (" [meta layer simulated]" if name.startswith("Meta-VQC") else ""), "inputs": sl,
             "test_macro_f1": macro(y_te, pr, labels=range(K)), "ci_low": lo, "ci_high": hi,
             "lgg_only_macro_f1": macro(y_te[lg], pr[lg], labels=[CLASSES.index(c) for c in LGG]),
             "balanced_acc": balanced_accuracy_score(y_te, pr)}
        if sl.startswith("(c)"):
            dlt, ci = paired_boot(y_te, pr, HW_PRED[(name, "(b) simulator, same models as hardware")])
            r.update({"hw_minus_same_models": dlt, "diff_ci_low": ci[0], "diff_ci_high": ci[1],
                      "same_prediction_pct": 100 * float((pr == HW_PRED[(name, "(b) simulator, same models as hardware")]).mean())})
        hw_rows.append(r)
FUSION_HW = pd.DataFrame(hw_rows)
FUSION_HW.to_csv(os.path.join(OUTPUT_DIR, "fusion_on_hardware.csv"), index=False)
display(FUSION_HW.round(3))
p = FUSION_HW[FUSION_HW["fusion (VQC base)"] == "Learned weights (global)"].set_index("inputs")
if len(p) == 3:
    a, b, c = p.iloc[0], p.iloc[1], p.iloc[2]
    print(f"• End-to-end hardware result: the learned-weight pool on the IBM-hardware outputs of all six modality VQCs reached "
          f"a test macro-F1 of {c.test_macro_f1:.3f} (95% CI {c.ci_low:.3f}–{c.ci_high:.3f}), against {b.test_macro_f1:.3f} for "
          f"the same models in noise-free simulation (difference {c.hw_minus_same_models:+.3f}, 95% CI {c.diff_ci_low:+.3f} to "
          f"{c.diff_ci_high:+.3f}; {c.same_prediction_pct:.1f}% of patients unchanged) and {a.test_macro_f1:.3f} with the full "
          f"simulated ensembles. LGG-only: {c.lgg_only_macro_f1:.3f} (hardware) vs {b.lgg_only_macro_f1:.3f}.")
print("Saved fusion_on_hardware.csv in", OUTPUT_DIR)

,fusion (VQC base),inputs,test_macro_f1,ci_low,ci_high,lgg_only_macro_f1,balanced_acc,hw_minus_same_models,diff_ci_low,diff_ci_high,same_prediction_pct
0,Learned weights (global),"(a) simulator, full ensembles",0.698,0.618,0.769,0.603,0.714,NaN,NaN,NaN,NaN
1,Learned weights (global),"(b) simulator, same models as hardware",0.698,0.618,0.769,0.603,0.714,NaN,NaN,NaN,NaN
2,Learned weights (global),(c) IBM hardware,0.694,0.613,0.764,0.597,0.715,-0.004,-0.036,0.023,95.968
3,Mean,"(a) simulator, full ensembles",0.715,0.634,0.786,0.641,0.736,NaN,NaN,NaN,NaN
4,Mean,"(b) simulator, same models as hardware",0.712,0.631,0.785,0.641,0.736,NaN,NaN,NaN,NaN
5,Mean,(c) IBM hardware,0.716,0.637,0.786,0.637,0.736,0.004,-0.032,0.042,94.355
6,Reliability-weighted,"(a) simulator, full ensembles",0.730,0.650,0.799,0.651,0.742,NaN,NaN,NaN,NaN
7,Reliability-weighted,"(b) simulator, same models as hardware",0.732,0.652,0.803,0.654,0.750,NaN,NaN,NaN,NaN
8,Reliability-weighted,(c) IBM hardware,0.715,0.635,0.786,0.631,0.737,-0.018,-0.045,0.000,96.774
9,Learned weights (per class),"(a) simulator, full ensembles",0.709,0.633,0.778,0.612,0.720,NaN,NaN,NaN,NaN


• End-to-end hardware result: the learned-weight pool on the IBM-hardware outputs of all six modality VQCs reached a test macro-F1 of 0.694 (95% CI 0.613–0.764), against 0.698 for the same models in noise-free simulation (difference -0.004, 95% CI -0.036 to +0.023; 96.0% of patients unchanged) and 0.698 with the full simulated ensembles. LGG-only: 0.597 (hardware) vs 0.603.
Saved fusion_on_hardware.csv in /kaggle/working
